In [ ]:
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/evaluation.csv')
data['Agent_Choice'] = data['Agent_Choice'].replace("A", "Compliance")
data['Agent_Choice'] = data['Agent_Choice'].replace("B", "Harmlessness")

roles = data['Role'].unique().tolist()
types = data['Type'].unique().tolist()
models = data['Model'].unique().tolist()

df = data[data['Model'] == models[0]].copy()


# Define the dropdown
dropdown_models = widgets.Dropdown(
    options=models,
    value=models[0],  # Default selection
    description='Model:',
)

# Define the dropdown
dropdown_roles = widgets.Dropdown(
    options=roles,
    value=roles[0],  # Default selection
    description='Role:',
)

# Define the dropdown
dropdown_types = widgets.Dropdown(
    options=types,
    value=types[0],  # Default selection
    description='Type:',
)

# Define update function
def update_data(selected_model):

    df = data[data['Model'] == selected_model].copy()

    # Get unique values for CA and HH
    CA = df['Compliance_Authority'].unique().tolist()
    HH = df['Harmlessness_Harm'].unique().tolist()
    HV = df['Harmlessness_Victim'].unique().tolist()

    # print(HV)
    # print(Role)
    # print(Type)

    # Create an empty matrix
    matrix_df = pd.DataFrame(index=HH, columns=CA)
    matrix_df = matrix_df.astype(float)  # Ensure DataFrame is float type
    matrix_df.fillna(0, inplace=True)  # Fill NaN values safely

    # Loop through records and update the matrix
    for index, row in df.iterrows():
        ca_value = row['Compliance_Authority']
        hh_value = row['Harmlessness_Harm']

        if row['Agent_Choice'] == 'Compliance':
            matrix_df.loc[hh_value, ca_value] += 1  # Increase the value

        elif row['Agent_Choice'] == 'Harmlessness':
            matrix_df.loc[hh_value, ca_value] -= 1  # Decrease the value

    # Separate positive and negative values for normalization
    positive_values = matrix_df[matrix_df > 0]
    negative_values = matrix_df[matrix_df < 0]

    # Create a copy for normalization and esure the DataFrame uses float type before modifying values
    normalized_matrix = matrix_df.copy().astype(float)

    # Normalize positive values separately
    pos_mask = matrix_df > 0
    if pos_mask.any().any():
        pos_min, pos_max = matrix_df[pos_mask].min().min(), matrix_df[pos_mask].max().max()
        normalized_matrix[pos_mask] = (matrix_df[pos_mask] - pos_min) / (pos_max - pos_min)

    # Normalize negative values separately
    neg_mask = matrix_df < 0
    if neg_mask.any().any():
        neg_min, neg_max = matrix_df[neg_mask].min().min(), matrix_df[neg_mask].max().max()
        normalized_matrix[neg_mask] = -(matrix_df[neg_mask] - neg_max) / (neg_min - neg_max)  # Keeping negative scaling

    # Create heatmap using matplotlib
    fig, ax = plt.subplots(figsize=(8, 6))
    cmap = plt.get_cmap("YlGnBu")

    # Display heatmap with color mapping
    heatmap = ax.imshow(normalized_matrix, cmap=cmap, aspect="auto")

    # Overlay numeric values
    for i in range(len(matrix_df.index)):
        for j in range(len(matrix_df.columns)):
            ax.text(j, i, f"{matrix_df.iloc[i, j]:.0f}", ha='center', va='center', color='black')

    # Add color bar
    cbar = plt.colorbar(heatmap)
    cbar.set_label("Prioritized Values")

    # Set custom labels for color bar
    cbar.set_ticks([-1, 1])  # Place labels at -1 and 1
    cbar.set_ticklabels(["Harmlessness", "Compliance"])  # Custom labels

    # Set tick labels
    ax.set_xticks(np.arange(len(matrix_df.columns)))
    ax.set_xticklabels(matrix_df.columns)
    ax.set_yticks(np.arange(len(matrix_df.index)))
    ax.set_yticklabels(matrix_df.index)

    # Label axes
    ax.set_xlabel("Compliance to Authority...")
    ax.set_ylabel("Harmlessness in Respect to...")
    ax.set_title(f"Heatmap of Agent Choices with model {selected_model}")

    print()
    plt.show()

# Create interactive widget
models_widget = widgets.interactive(update_data, selected_model=dropdown_models)
#types_widget = widgets.interactive(update_data, selected_type=dropdown_types)
#roles_widget = widgets.interactive(update_data, selected_role=dropdown_roles)

# Display the dropdown in Colab
print("Select the model, type and role for which we want to plot the results:\n")
display(models_widget)
#display(types_widget)
#display(roles_widget)


Select the model, type and role for which we want to plot the results:



interactive(children=(Dropdown(description='Model:', options=('Qwen/Qwen3-235B-A22B', 'deepseek-ai/DeepSeek-V3…